# Bài Kiểm Tra Thực Hành 2 — Phân Loại Email Spam (MLOps)

**Họ tên:** Ngô Hồng Thông
**Dataset:** `spamDataset.csv`
**Mục tiêu:** Xây dựng pipeline MLOps đầy đủ 5 yêu cầu

| # | Yêu cầu | Ghi chú |
|---|---|---|
| 1 | Data Pipeline | Làm sạch + TF-IDF |
| 2 | Training & Tracking | Naive Bayes + XGBoost + MLflow |
| 3 | Model Registry | Staging / Production + so sánh |
| 4 | Deployment | FastAPI `/predict` |
| 5 | CI/CD & Monitoring | GitHub Actions + Drift detection |


---
## Phần 0 — Cài đặt môi trường


In [ ]:
# [0.1] Cai dat thu vien can thiet cho Colab
# Note: Phai dung numpy 1.x vi MLflow + xgboost chua tuong thich numpy 2.x

!pip -q uninstall -y numpy
!pip -q install numpy==1.26.4 mlflow scikit-learn xgboost \
    fastapi "uvicorn[standard]" pydantic requests tabulate

# Tai cloudflared de expose service ra internet
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 \
     -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared

print("[OK] Da cai dat xong — chay cell 0.2 de RESTART runtime")


In [ ]:
# [0.2] RESTART runtime sau khi cai numpy moi
# Sau khi cell nay chay, runtime se chet — chay tiep tu Phan 1
import os; os.kill(os.getpid(), 9)


---
## Phần 1 — Cấu hình project

Gom toàn bộ tham số / đường dẫn vào một class `CFG` để dễ thay đổi.


In [ ]:
# [1.1] Class cau hinh trung tam
import os

class CFG:
    # Thong tin sinh vien
    STUDENT_ID  = "ngohongthong"
    PROJECT_DIR = f"/content/{STUDENT_ID}"

    # 5 thu muc tuong ung 5 yeu cau cua de bai
    DIR_PIPELINE = f"{PROJECT_DIR}/pipeline_du_lieu"
    DIR_TRAIN    = f"{PROJECT_DIR}/huan_luyen_va_track"
    DIR_REGISTRY = f"{PROJECT_DIR}/dang_ky_model"
    DIR_DEPLOY   = f"{PROJECT_DIR}/trien_khai_api"
    DIR_OPS      = f"{PROJECT_DIR}/cicd_va_giam_sat"

    # MLflow config
    MLFLOW_DB        = "sqlite:////content/mlflow.db"
    MLFLOW_ARTIFACTS = "/content/mlflow_artifacts"
    EXPERIMENT_NAME  = "phan_loai_email_spam"
    MODEL_NAME       = "EmailSpamModel"

    # Training config
    TFIDF_FEATURES   = 5000
    TFIDF_NGRAM      = (1, 2)
    SEED             = 42

    # API config
    API_PORT     = 8000
    MLFLOW_PORT  = 5000

    # Monitoring
    LOG_FILE     = f"{DIR_OPS}/api_log.jsonl"
    DRIFT_K      = 2.0   # Nguong drift = 2*std

# Tao tat ca thu muc
for path in [CFG.DIR_PIPELINE, CFG.DIR_TRAIN, CFG.DIR_REGISTRY,
             CFG.DIR_DEPLOY, CFG.DIR_OPS, CFG.MLFLOW_ARTIFACTS]:
    os.makedirs(path, exist_ok=True)

print(f"[OK] Project: {CFG.PROJECT_DIR}")
print(f"[OK] Da tao {5} thu muc con")


---
## Yêu cầu 1 — Data Pipeline

**Mục tiêu:** Đọc CSV → làm sạch → TF-IDF → lưu artifacts

**Đầu ra:** 3 file trong `pipeline_du_lieu/`:
- `spam_clean.csv` — dataset đã làm sạch
- `vec.pkl` — TF-IDF vectorizer
- `splits.pkl` — bộ train/val/test


In [ ]:
# [Y1.1] Upload dataset spamDataset.csv tu may local
from google.colab import files
print("Vui long upload file spamDataset.csv:")
uploaded = files.upload()
CSV_FILE = list(uploaded.keys())[0]
print(f"[OK] Da upload: {CSV_FILE}")


In [ ]:
# [Y1.2] Doc CSV va lam sach co ban
import pandas as pd
import re

# Doc file CSV (encoding latin-1 vi co ky tu dac biet)
raw = pd.read_csv(CSV_FILE, encoding='latin-1', usecols=['v1', 'v2'])
raw.columns = ['nhan', 'noi_dung']
raw = raw.dropna().drop_duplicates().reset_index(drop=True)

# Map nhan: ham=0, spam=1
raw['nhan_so'] = raw['nhan'].map({'ham': 0, 'spam': 1})

# Ham lam sach text
def chuan_hoa(s: str) -> str:
    s = str(s).lower()
    s = re.sub(r'http\S+|www\.\S+', ' ', s)   # loai URL
    s = re.sub(r'\d+', ' ', s)                  # loai chu so
    s = re.sub(r'[^a-z\s]', ' ', s)             # giu chu cai va space
    s = re.sub(r'\s+', ' ', s).strip()
    return s

raw['noi_dung_sach'] = raw['noi_dung'].apply(chuan_hoa)

print(f"[OK] Tong so mau: {len(raw)}")
print(f"[OK] Phan bo nhan:")
print(raw['nhan'].value_counts().to_string())
print()
print("[OK] 5 mau dau:")
print(raw[['nhan', 'noi_dung_sach']].head().to_string(index=False))


In [ ]:
# [Y1.3] Tach tap train/val/test theo ti le 60/20/20
from sklearn.model_selection import train_test_split

X_full = raw['noi_dung_sach'].values
y_full = raw['nhan_so'].values

# Tach lan 1: 80% temp + 20% test
X_tmp, X_test, y_tmp, y_test = train_test_split(
    X_full, y_full,
    test_size=0.2, random_state=CFG.SEED, stratify=y_full,
)
# Tach lan 2: trong 80% temp -> 75% train + 25% val (= 60/20 cua tong)
X_train, X_val, y_train, y_val = train_test_split(
    X_tmp, y_tmp,
    test_size=0.25, random_state=CFG.SEED, stratify=y_tmp,
)

print(f"[OK] Train: {len(X_train):>4} mau ({len(X_train)/len(X_full):.0%})")
print(f"[OK] Val  : {len(X_val):>4} mau ({len(X_val)/len(X_full):.0%})")
print(f"[OK] Test : {len(X_test):>4} mau ({len(X_test)/len(X_full):.0%})")


In [ ]:
# [Y1.4] Bien doi van ban thanh vector TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer

# Dung n-gram (1,2) de bat duoc cum tu pho bien
vec = TfidfVectorizer(
    max_features = CFG.TFIDF_FEATURES,
    ngram_range  = CFG.TFIDF_NGRAM,
    stop_words   = 'english',
    min_df       = 2,
)

Xv_train = vec.fit_transform(X_train)
Xv_val   = vec.transform(X_val)
Xv_test  = vec.transform(X_test)

print(f"[OK] TF-IDF shape: {Xv_train.shape}")
print(f"[OK] So feature thuc te: {len(vec.get_feature_names_out())}")


In [ ]:
# [Y1.5] Luu artifacts ra disk
import pickle

# Luu DataFrame da lam sach
raw.to_csv(f"{CFG.DIR_PIPELINE}/spam_clean.csv", index=False)

# Luu vectorizer
with open(f"{CFG.DIR_PIPELINE}/vec.pkl", "wb") as f:
    pickle.dump(vec, f)

# Luu splits (ca raw text de tinh drift sau nay)
splits = dict(
    Xv_train=Xv_train, Xv_val=Xv_val, Xv_test=Xv_test,
    y_train=y_train,   y_val=y_val,   y_test=y_test,
    X_train_raw=X_train, X_val_raw=X_val, X_test_raw=X_test,
)
with open(f"{CFG.DIR_PIPELINE}/splits.pkl", "wb") as f:
    pickle.dump(splits, f)

# Liet ke file da tao
import os
print(f"[OK] Da luu vao {CFG.DIR_PIPELINE}:")
for f in sorted(os.listdir(CFG.DIR_PIPELINE)):
    size = os.path.getsize(f"{CFG.DIR_PIPELINE}/{f}")
    print(f"  - {f:25s} ({size:>7,} bytes)")


---
## Yêu cầu 2 — Model Training & Tracking

**Train 2 mô hình:** Naive Bayes (baseline) và XGBoost (mạnh hơn)
**Đánh giá:** Precision, Recall, F1 trên cả Validation và Test
**Track bằng MLflow:** log Parameters, Metrics, Model Artifacts


In [ ]:
# [Y2.1] Khoi tao MLflow tracking
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature

mlflow.set_tracking_uri(CFG.MLFLOW_DB)
mlflow.set_experiment(CFG.EXPERIMENT_NAME)

print(f"[OK] Tracking URI : {mlflow.get_tracking_uri()}")
print(f"[OK] Experiment   : {CFG.EXPERIMENT_NAME}")
print(f"[OK] MLflow ver   : {mlflow.__version__}")


In [ ]:
# [Y2.2] Ham phu: tinh metric va in classification report
import numpy as np
from sklearn.metrics import (
    precision_score, recall_score, f1_score, accuracy_score,
    classification_report,
)


def tinh_metric(y_true, y_pred, prefix: str) -> dict:
    """Tinh 4 metric: precision, recall, f1, accuracy."""
    return {
        f"{prefix}_precision": precision_score(y_true, y_pred),
        f"{prefix}_recall":    recall_score(y_true, y_pred),
        f"{prefix}_f1":        f1_score(y_true, y_pred),
        f"{prefix}_accuracy":  accuracy_score(y_true, y_pred),
    }


def in_ket_qua(ten_model: str, m_val: dict, m_test: dict, y_true, y_pred):
    """In ket qua train cua mot model gon gang."""
    print(f"\n{'='*55}")
    print(f"  Model: {ten_model}")
    print(f"{'='*55}")
    print(f"  [VAL ] P={m_val['val_precision']:.4f}  "
          f"R={m_val['val_recall']:.4f}  "
          f"F1={m_val['val_f1']:.4f}")
    print(f"  [TEST] P={m_test['test_precision']:.4f}  "
          f"R={m_test['test_recall']:.4f}  "
          f"F1={m_test['test_f1']:.4f}")
    print(f"\n{classification_report(y_true, y_pred, target_names=['ham','spam'])}")


In [ ]:
# [Y2.3] Train Naive Bayes (model A)
from sklearn.naive_bayes import MultinomialNB

with mlflow.start_run(run_name="NB_baseline") as run_a:
    nb_params = {"loai_model": "MultinomialNB", "alpha": 1.0}
    nb = MultinomialNB(alpha=1.0)
    nb.fit(Xv_train, y_train)

    pred_val  = nb.predict(Xv_val)
    pred_test = nb.predict(Xv_test)

    m_val  = tinh_metric(y_val,  pred_val,  "val")
    m_test = tinh_metric(y_test, pred_test, "test")

    # Log MLflow
    mlflow.log_params(nb_params)
    mlflow.log_params({"tfidf_features": CFG.TFIDF_FEATURES,
                       "ngram_range":   str(CFG.TFIDF_NGRAM)})
    mlflow.log_metrics({**m_val, **m_test})
    mlflow.sklearn.log_model(
        sk_model=nb,
        artifact_path="model",
        registered_model_name=CFG.MODEL_NAME,
        signature=infer_signature(Xv_test, pred_test),
    )

    in_ket_qua("Naive Bayes (alpha=1.0)", m_val, m_test, y_test, pred_test)
    RUN_NB = run_a.info.run_id
    print(f"  Run ID: {RUN_NB[:8]}")


In [ ]:
# [Y2.4] Train XGBoost (model B)
from xgboost import XGBClassifier

with mlflow.start_run(run_name="XGB_boosted") as run_b:
    xgb_params = {
        "loai_model"    : "XGBClassifier",
        "n_estimators"  : 250,
        "max_depth"     : 5,
        "learning_rate" : 0.12,
        "subsample"     : 0.9,
    }
    xgb = XGBClassifier(
        n_estimators  = xgb_params["n_estimators"],
        max_depth     = xgb_params["max_depth"],
        learning_rate = xgb_params["learning_rate"],
        subsample     = xgb_params["subsample"],
        eval_metric   = "logloss",
        random_state  = CFG.SEED,
    )
    xgb.fit(Xv_train, y_train)

    pred_val  = xgb.predict(Xv_val)
    pred_test = xgb.predict(Xv_test)

    m_val  = tinh_metric(y_val,  pred_val,  "val")
    m_test = tinh_metric(y_test, pred_test, "test")

    mlflow.log_params(xgb_params)
    mlflow.log_params({"tfidf_features": CFG.TFIDF_FEATURES,
                       "ngram_range":   str(CFG.TFIDF_NGRAM)})
    mlflow.log_metrics({**m_val, **m_test})
    mlflow.sklearn.log_model(
        sk_model=xgb,
        artifact_path="model",
        registered_model_name=CFG.MODEL_NAME,
        signature=infer_signature(Xv_test, pred_test),
    )

    in_ket_qua("XGBoost (250 trees)", m_val, m_test, y_test, pred_test)
    RUN_XGB = run_b.info.run_id
    print(f"  Run ID: {RUN_XGB[:8]}")


---
## Yêu cầu 3 — Model Registry

- Đăng ký 2 version vào Registry (đã đăng ký tự động ở Yêu cầu 2)
- Gán **Staging** cho version baseline, **Production** cho version tốt hơn
- So sánh và **giải thích vì sao chọn version đó**


In [ ]:
# [Y3.1] Lay danh sach run, sap xep theo test_f1 giam dan
from mlflow.tracking import MlflowClient
from tabulate import tabulate

cli = MlflowClient(tracking_uri=CFG.MLFLOW_DB)

exp = cli.get_experiment_by_name(CFG.EXPERIMENT_NAME)
all_runs = cli.search_runs(
    experiment_ids=[exp.experiment_id],
    order_by=["metrics.test_f1 DESC"],
)

# Bang tom tat
bang = []
for r in all_runs:
    m = r.data.metrics
    bang.append([
        r.info.run_id[:8],
        r.info.run_name,
        f"{m.get('val_f1', 0):.4f}",
        f"{m.get('test_precision', 0):.4f}",
        f"{m.get('test_recall', 0):.4f}",
        f"{m.get('test_f1', 0):.4f}",
        f"{m.get('test_accuracy', 0):.4f}",
    ])

print(tabulate(
    bang,
    headers=["RunID", "Ten run", "Val F1", "Test P", "Test R", "Test F1", "Test Acc"],
    tablefmt="grid",
))

best, worse = all_runs[0], all_runs[1]
print(f"\n[BEST]  {best.info.run_name}  (test_f1={best.data.metrics['test_f1']:.4f})")
print(f"[OTHER] {worse.info.run_name} (test_f1={worse.data.metrics['test_f1']:.4f})")


In [ ]:
# [Y3.2] Map run_id → version trong Registry
versions_all = cli.search_model_versions(f"name='{CFG.MODEL_NAME}'")
versions_all = sorted(versions_all, key=lambda v: int(v.version))

ver_best  = next(v.version for v in versions_all if v.run_id == best.info.run_id)
ver_other = next(v.version for v in versions_all if v.run_id == worse.info.run_id)

print(f"[INFO] Tong so version trong Registry: {len(versions_all)}")
print(f"[INFO] Version cua model tot nhat   : v{ver_best}")
print(f"[INFO] Version cua model con lai    : v{ver_other}")


In [ ]:
# [Y3.3] Gan stage Staging va Production
# MLflow 3.x dung alias, MLflow 2.x dung stage — thu ca hai

def gan_stage(version: str, stage_name: str):
    """Gan stage cho mot version, tu dong fallback giua alias va stage."""
    try:
        cli.set_registered_model_alias(CFG.MODEL_NAME, stage_name, version)
        print(f"[OK] Version {version} → alias '{stage_name}'")
    except AttributeError:
        cli.transition_model_version_stage(
            name=CFG.MODEL_NAME, version=version,
            stage=stage_name, archive_existing_versions=False,
        )
        print(f"[OK] Version {version} → stage '{stage_name}'")

gan_stage(ver_other, "Staging")
gan_stage(ver_best,  "Production")

# Tag them metadata cho version tot nhat
cli.set_model_version_tag(CFG.MODEL_NAME, ver_best, "test_f1",
                          str(round(best.data.metrics["test_f1"], 4)))
cli.set_model_version_tag(CFG.MODEL_NAME, ver_best, "trang_thai", "production")

# In trang thai cuoi cung
print(f"\n[REGISTRY] {CFG.MODEL_NAME}:")
for v in cli.search_model_versions(f"name='{CFG.MODEL_NAME}'"):
    aliases = getattr(v, "aliases", []) or []
    stage   = getattr(v, "current_stage", "None")
    print(f"  v{v.version} | stage={stage:<15} aliases={aliases} | run={v.run_id[:8]}")


In [ ]:
# [Y3.4] So sanh chi tiet 2 version + giai thich
print("="*60)
print("  SO SANH 2 MODEL VERSION")
print("="*60)

mb = best.data.metrics
mo = worse.data.metrics

print(f"\n  v{ver_best:<3} ({best.info.run_name})  [PRODUCTION]")
print(f"    Precision = {mb['test_precision']:.4f}")
print(f"    Recall    = {mb['test_recall']:.4f}")
print(f"    F1        = {mb['test_f1']:.4f}")

print(f"\n  v{ver_other:<3} ({worse.info.run_name})  [STAGING]")
print(f"    Precision = {mo['test_precision']:.4f}")
print(f"    Recall    = {mo['test_recall']:.4f}")
print(f"    F1        = {mo['test_f1']:.4f}")

print("\n" + "="*60)
print("  GIAI THICH (3-5 dong)")
print("="*60)
print(f"""
1. Chon **{best.info.run_name}** lam Production vi co Test F1 cao nhat
   ({mb['test_f1']:.4f} vs {mo['test_f1']:.4f}).
2. F1 cao the hien su can bang giua Precision (khong nham ham→spam)
   va Recall (khong bo sot spam) — cuc ki quan trong cho he thong loc email.
3. Model con lai duoc giu o stage **Staging** de fallback
   neu Production gap su co hoac de A/B test sau nay.
4. Nho he thong alias/stage cua MLflow Registry, ta co the swap version
   ma khong can deploy lai code — chi can chuyen alias.
""")

# Luu thong tin de yeu cau 4 dung
import json
with open(f"{CFG.DIR_REGISTRY}/registry_info.json", "w") as f:
    json.dump({
        "production_version": ver_best,
        "staging_version":    ver_other,
        "best_run_name":      best.info.run_name,
        "best_test_f1":       mb["test_f1"],
    }, f, indent=2, ensure_ascii=False)

print(f"[OK] Da luu registry_info.json")


---
## Yêu cầu 4 — Model Deployment (FastAPI)

- Endpoint `/predict` nhận text → trả về label (`ham`/`spam`)
- Load model trực tiếp từ MLflow Registry (alias `Production`) — **được điểm cộng**
- Test bằng `requests` (hoặc curl/Postman với URL Cloudflare)


In [ ]:
# [Y4.1] Viet code FastAPI ra file rieng (de uvicorn import duoc)
%%writefile /content/api_server.py
"""FastAPI server — Phan loai email spam/ham.

- GET  /         : trang chu HTML
- GET  /health   : kiem tra trang thai
- POST /predict  : du doan label cho text
- GET  /drift    : kiem tra data drift
"""
import os, re, json, pickle
from datetime import datetime
from typing import Literal

import numpy as np
import mlflow.sklearn
from fastapi import FastAPI, HTTPException
from fastapi.responses import HTMLResponse
from pydantic import BaseModel, Field

# Cau hinh (giong CFG nhung viet rieng vi script chay doc lap)
PROJECT     = "/content/ngohongthong"
MLFLOW_DB   = "sqlite:////content/mlflow.db"
MODEL_NAME  = "EmailSpamModel"
VEC_PATH    = f"{PROJECT}/pipeline_du_lieu/vec.pkl"
SPLITS_PATH = f"{PROJECT}/pipeline_du_lieu/splits.pkl"
LOG_PATH    = f"{PROJECT}/cicd_va_giam_sat/api_log.jsonl"

os.environ["MLFLOW_TRACKING_URI"] = MLFLOW_DB
os.makedirs(os.path.dirname(LOG_PATH), exist_ok=True)


# === Lazy-load resource (chi load 1 lan o request dau) ===
_state = {"model": None, "vec": None}


def _load_vec():
    if _state["vec"] is None:
        with open(VEC_PATH, "rb") as f:
            _state["vec"] = pickle.load(f)
        print("[INFO] TF-IDF vectorizer da nap")
    return _state["vec"]


def _load_model():
    if _state["model"] is not None:
        return _state["model"]
    # Thu nhieu URI: alias (MLflow 3) -> stage (MLflow 2) -> version moi nhat
    for uri in [f"models:/{MODEL_NAME}@Production",
                f"models:/{MODEL_NAME}/Production"]:
        try:
            _state["model"] = mlflow.sklearn.load_model(uri)
            print(f"[INFO] Model da nap tu: {uri}")
            return _state["model"]
        except Exception:
            pass
    # Fallback: lay version cao nhat
    from mlflow.tracking import MlflowClient
    cli = MlflowClient()
    vs = cli.search_model_versions(f"name='{MODEL_NAME}'")
    last = sorted(vs, key=lambda v: int(v.version))[-1]
    _state["model"] = mlflow.sklearn.load_model(f"models:/{MODEL_NAME}/{last.version}")
    print(f"[INFO] Fallback: load version {last.version}")
    return _state["model"]


def _clean(s: str) -> str:
    s = str(s).lower()
    s = re.sub(r'http\S+|www\.\S+', ' ', s)
    s = re.sub(r'\d+',                ' ', s)
    s = re.sub(r'[^a-z\s]',           ' ', s)
    return re.sub(r'\s+', ' ', s).strip()


def _ghi_log(text: str, label: str, conf: float):
    """Ghi mot dong JSONL cho monitoring."""
    rec = {
        "thoi_diem": datetime.now().isoformat(timespec="seconds"),
        "do_dai":    len(text),
        "so_tu":     len(text.split()),
        "ket_qua":   label,
        "do_tin":    round(conf, 4),
    }
    with open(LOG_PATH, "a") as f:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")


# === FastAPI app ===
app = FastAPI(
    title="Email Spam Classifier",
    version="2.0",
    description="Phan loai email spam/ham — load model tu MLflow Registry",
)


class YeuCau(BaseModel):
    text: str = Field(..., min_length=1, description="Noi dung email")


class KetQua(BaseModel):
    nhan_du_doan : Literal["ham", "spam"]
    do_tin       : float
    nhan_so      : int


@app.get("/", response_class=HTMLResponse)
def trang_chu():
    return """
    <html><head><title>Spam Classifier</title>
    <style>body{font-family:sans-serif;max-width:680px;margin:3em auto;padding:1em;
                background:#fafafa}
           h1{color:#0f766e} code{background:#e2e8f0;padding:2px 6px;border-radius:4px}
           a{color:#0f766e}</style></head><body>
    <h1>Email Spam Classifier</h1>
    <p>Backend: <strong>FastAPI</strong> + <strong>MLflow Registry</strong></p>
    <ul>
      <li><code>GET /health</code></li>
      <li><code>POST /predict</code> body: <code>{"text": "..."}</code></li>
      <li><code>GET /drift</code></li>
      <li><a href="/docs">Swagger UI</a></li>
    </ul></body></html>"""


@app.get("/health")
def kiem_tra():
    try:
        _load_model(); _load_vec()
        return {"trang_thai": "ok", "model": MODEL_NAME}
    except Exception as e:
        raise HTTPException(500, str(e))


@app.post("/predict", response_model=KetQua)
def du_doan(req: YeuCau):
    model, vec = _load_model(), _load_vec()
    X = vec.transform([_clean(req.text)])

    nhan_so = int(model.predict(X)[0])
    nhan    = "spam" if nhan_so == 1 else "ham"

    do_tin = float(model.predict_proba(X)[0][nhan_so]) \
             if hasattr(model, "predict_proba") else 1.0

    _ghi_log(req.text, nhan, do_tin)
    return KetQua(nhan_du_doan=nhan, do_tin=round(do_tin, 4), nhan_so=nhan_so)


@app.get("/drift")
def kiem_tra_drift():
    """Drift = chenh lech do dai trung binh email so voi tap train."""
    with open(SPLITS_PATH, "rb") as f:
        sp = pickle.load(f)

    do_dai_train = [len(t) for t in sp["X_train_raw"]]
    mu_train     = float(np.mean(do_dai_train))
    sd_train     = float(np.std(do_dai_train))

    if not os.path.exists(LOG_PATH):
        return {"trang_thai": "chua_co_log"}

    do_dai_moi = []
    with open(LOG_PATH) as f:
        for line in f:
            do_dai_moi.append(json.loads(line)["do_dai"])

    if len(do_dai_moi) < 5:
        return {"trang_thai": "thieu_du_lieu", "so_request": len(do_dai_moi)}

    mu_moi = float(np.mean(do_dai_moi))
    score  = abs(mu_moi - mu_train) / (sd_train + 1e-8)
    drift  = score > 2.0

    return {
        "trang_thai":     "co_drift" if drift else "binh_thuong",
        "do_dai_tb_train": round(mu_train, 2),
        "do_dai_tb_moi":   round(mu_moi,   2),
        "diem_drift":      round(score,    4),
        "nguong":          2.0,
        "tong_request":    len(do_dai_moi),
        "khuyen_nghi":     "Nen retrain model!" if drift else "Model on dinh.",
    }


In [ ]:
# [Y4.2] Khoi dong MLflow server + FastAPI + Cloudflare tunnel
import subprocess, threading, time, urllib.request, urllib.error, re, os

os.environ["MLFLOW_ENABLE_PROXY_FIX"] = "true"
os.environ["GUNICORN_CMD_ARGS"]       = "--forwarded-allow-ips='*'"

# Container chua URL public (de cell sau dung)
URLS = {"mlflow": None, "api": None}


def _stream(proc, tag):
    """Doc stdout cua subprocess theo realtime."""
    for line in iter(proc.stdout.readline, b""):
        print(tag + line.decode(errors="replace").rstrip())


def _doi_port(port: int, timeout: int = 60) -> bool:
    """Chu cho den khi port san sang nhan request."""
    t0 = time.time()
    while time.time() - t0 < timeout:
        try:
            urllib.request.urlopen(f"http://localhost:{port}/", timeout=2)
            return True
        except urllib.error.HTTPError:
            return True   # 404/405 cung tinh la up
        except Exception:
            time.sleep(1)
    return False


def _mo_tunnel(port: int, ten: str):
    """Mo cloudflared tunnel cho mot port, parse URL public."""
    proc = subprocess.Popen(
        ["/usr/local/bin/cloudflared", "tunnel",
         "--url", f"http://localhost:{port}", "--no-autoupdate"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    )
    for line in iter(proc.stdout.readline, b""):
        txt = line.decode(errors="replace").rstrip()
        m = re.search(r"https://[a-z0-9\-]+\.trycloudflare\.com", txt)
        if m and not URLS[ten]:
            URLS[ten] = m.group(0)
            print(f"\n[CF] {ten.upper()}: {URLS[ten]}")


# 1) Khoi dong MLflow server
ml_proc = subprocess.Popen(
    ["mlflow", "server",
     "--host", "0.0.0.0", "--port", str(CFG.MLFLOW_PORT),
     "--backend-store-uri",     CFG.MLFLOW_DB,
     "--default-artifact-root", f"file://{CFG.MLFLOW_ARTIFACTS}",
     "--allowed-hosts",         "*"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, env={**os.environ},
)
threading.Thread(target=_stream, args=(ml_proc, "[ml] "), daemon=True).start()
print("Cho MLflow server san sang...", end="", flush=True)
print(" ready" if _doi_port(CFG.MLFLOW_PORT) else " timeout")

# 2) Khoi dong FastAPI
api_proc = subprocess.Popen(
    ["python", "-m", "uvicorn", "api_server:app",
     "--host", "0.0.0.0", "--port", str(CFG.API_PORT)],
    cwd="/content",
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, env={**os.environ},
)
threading.Thread(target=_stream, args=(api_proc, "[api] "), daemon=True).start()
print("Cho FastAPI san sang...", end="", flush=True)
print(" ready" if _doi_port(CFG.API_PORT) else " timeout")

# 3) Mo 2 tunnel
threading.Thread(target=_mo_tunnel, args=(CFG.MLFLOW_PORT, "mlflow"), daemon=True).start()
time.sleep(2)
threading.Thread(target=_mo_tunnel, args=(CFG.API_PORT,    "api"   ), daemon=True).start()

print("\nDang lay URL Cloudflare (~60s)...")
for _ in range(40):
    time.sleep(2)
    if URLS["mlflow"] and URLS["api"]:
        break

print("\n" + "="*65)
print(f"  MLflow UI : {URLS['mlflow'] or '(timeout)'}")
print(f"  API       : {URLS['api']    or '(timeout)'}")
if URLS["api"]:
    print(f"  Docs      : {URLS['api']}/docs")
    print(f"  Drift     : {URLS['api']}/drift")
print("="*65)


In [ ]:
# [Y4.3] Test API voi nhieu mau email khac nhau
import requests, time
from tabulate import tabulate

API = URLS.get("api") or f"http://localhost:{CFG.API_PORT}"

# Doi API san sang
for _ in range(5):
    try:
        if requests.get(f"{API}/health", timeout=10).status_code == 200:
            break
    except Exception:
        time.sleep(3)

# Bo mau test (8 mau, 4 ham + 4 spam ro rang)
mau_test = [
    "Sounds good, see you at 7pm tonight!",
    "URGENT! Your account will be suspended. Click http://verify.scam now!",
    "Mom, can you pick up some milk on your way home?",
    "WIN a brand new iPhone 15! Reply YES to claim. Limited offer.",
    "Meeting moved to conference room B. Bring your laptop.",
    "Free Viagra! 70% discount! Order now and get free shipping worldwide.",
    "Thanks for the help yesterday, really appreciated it.",
    "Congratulations! You have been selected for a $500 Walmart gift card.",
]

ket_qua = []
for txt in mau_test:
    try:
        r = requests.post(f"{API}/predict", json={"text": txt}, timeout=15)
        d = r.json()
        ket_qua.append([
            d["nhan_du_doan"].upper(),
            f"{d['do_tin']:.2%}",
            txt[:60] + ("..." if len(txt) > 60 else ""),
        ])
    except Exception as e:
        ket_qua.append(["ERR", "-", str(e)[:60]])

print(f"API: {API}\n")
print(tabulate(ket_qua,
               headers=["Nhan", "Do tin", "Noi dung email"],
               tablefmt="rounded_grid"))


**Cách test khác bằng curl** (chạy trong terminal hoặc cell mới):

```bash
curl -X POST <API_URL>/predict \
  -H "Content-Type: application/json" \
  -d '{"text": "Free entry to win an iPhone now!"}'
```


---
## Yêu cầu 5 — CI/CD & Monitoring

3 phần nhỏ:
- **5a.** Xem prediction log + chạy drift detection
- **5b.** Sinh file `.github/workflows/mlops.yml` cho GitHub Actions
- **5c.** Viết unit test cho drift logic
- **5d.** Đề xuất khi nào retrain model


In [ ]:
# [Y5.1] Xem prediction log da ghi
import json, os
import pandas as pd

if os.path.exists(CFG.LOG_FILE):
    with open(CFG.LOG_FILE) as f:
        ds = pd.DataFrame([json.loads(l) for l in f])
    print(f"[OK] Tong so prediction da log: {len(ds)}")
    print(f"[OK] Phan bo nhan: {ds['ket_qua'].value_counts().to_dict()}")
    print()
    print(ds.tail(8).to_string(index=False))
else:
    print("[WARN] Chua co log — chay cell Y4.3 truoc")


In [ ]:
# [Y5.2] Goi endpoint /drift de kiem tra data drift
import requests, json

r = requests.get(f"{API}/drift", timeout=10)
result = r.json()

print(json.dumps(result, indent=2, ensure_ascii=False))
print()

if result.get("trang_thai") == "co_drift":
    print("[ALERT] Phat hien data drift — can retrain!")
elif result.get("trang_thai") == "binh_thuong":
    print("[OK] Du lieu on dinh, chua can retrain.")


In [ ]:
# [Y5.3] Sinh file CI/CD GitHub Actions
import os

# Tao thu muc theo dung chuan GitHub
GH_DIR = f"{CFG.DIR_OPS}/.github/workflows"
os.makedirs(GH_DIR, exist_ok=True)

cicd_yaml = """name: MLOps Pipeline cho Spam Classifier

# ---- Khi nao chay? ----
on:
  push:
    branches: [main]            # Push len main
  pull_request:
    branches: [main]            # Mo PR vao main
  workflow_dispatch:            # Cho phep chay tay

jobs:

  # =========================================
  # JOB 1: Chay data pipeline + train + test
  # =========================================
  pipeline:
    name: "Data + Train + Test"
    runs-on: ubuntu-latest

    steps:
      - name: Checkout repo
        uses: actions/checkout@v4

      - name: Cai Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.10"
          cache: "pip"

      - name: Cai dependencies
        run: |
          python -m pip install --upgrade pip
          pip install pandas scikit-learn xgboost mlflow \\
              fastapi uvicorn pydantic jupyter nbconvert pytest

      - name: Step 1 — Run data pipeline
        run: |
          jupyter nbconvert --to notebook --execute \\
              pipeline_du_lieu/data_pipeline.ipynb

      - name: Step 2 — Train model
        run: |
          jupyter nbconvert --to notebook --execute \\
              huan_luyen_va_track/train.ipynb

      - name: Step 3 — Run unit tests
        run: pytest cicd_va_giam_sat/tests -v

      - name: Upload artifacts
        if: always()
        uses: actions/upload-artifact@v4
        with:
          name: mlruns
          path: mlruns/
"""

with open(f"{GH_DIR}/mlops.yml", "w") as f:
    f.write(cicd_yaml)

print(f"[OK] Da tao: {GH_DIR}/mlops.yml")
print(f"[OK] Kich thuoc: {os.path.getsize(f'{GH_DIR}/mlops.yml'):,} bytes")
print()
print("--- 30 dong dau ---")
print("\n".join(cicd_yaml.splitlines()[:30]))


In [ ]:
# [Y5.4] Sinh unit test cho drift detection
import os, subprocess

TEST_DIR = f"{CFG.DIR_OPS}/tests"
os.makedirs(TEST_DIR, exist_ok=True)

# File __init__.py rong
open(f"{TEST_DIR}/__init__.py", "w").close()

# File test
test_code = '''"""Unit test cho ham phat hien data drift."""
import numpy as np


def kiem_tra_drift(do_dai_moi, mu_baseline, sd_baseline, k=2.0) -> bool:
    """Phat hien drift dua tren do dai trung binh.

    Logic: |mean(moi) - mu_baseline| > k * sd_baseline
    """
    if not do_dai_moi:
        return False
    return abs(np.mean(do_dai_moi) - mu_baseline) > k * sd_baseline


# ===== Cac kich ban test =====

def test_du_lieu_giong_baseline_thi_khong_drift():
    assert kiem_tra_drift([100, 102, 98, 101], mu_baseline=100, sd_baseline=20) is False

def test_du_lieu_lech_xa_thi_co_drift():
    assert kiem_tra_drift([300, 310, 305], mu_baseline=100, sd_baseline=20) is True

def test_input_rong_tra_ve_false():
    assert kiem_tra_drift([], mu_baseline=100, sd_baseline=20) is False

def test_threshold_lon_thi_khong_drift():
    # Voi k=10 thi 200 cung khong dat nguong (10*20=200)
    assert kiem_tra_drift([200, 210], mu_baseline=100, sd_baseline=20, k=10) is False
'''

with open(f"{TEST_DIR}/test_drift.py", "w") as f:
    f.write(test_code)

print(f"[OK] Da tao: {TEST_DIR}/test_drift.py")
print()

# Chay test luon de chac
print("Chay pytest...")
out = subprocess.run(
    ["python", "-m", "pytest", f"{TEST_DIR}/test_drift.py", "-v"],
    capture_output=True, text=True,
)
print(out.stdout)
if out.returncode:
    print(out.stderr)


### Khi nào nên retrain model?

**5 trường hợp cần retrain:**

| # | Tình huống | Dấu hiệu |
|---|---|---|
| 1 | **Data drift** | Endpoint `/drift` trả `co_drift` (score > 2.0) |
| 2 | **Performance suy giảm** | Test F1 trên dữ liệu mới giảm > 5% so với baseline |
| 3 | **Concept drift** | Loại spam mới xuất hiện (vd: phishing kiểu mới) mà model cũ không nhận diện được |
| 4 | **Định kỳ** | Sau 1-3 tháng, gom đủ dữ liệu mới (~10% kích thước tập train ban đầu) |
| 5 | **Phản hồi từ user** | Tỉ lệ false positive/negative cao (user đánh dấu lại label) |


---
## Phần cuối — Tổng kết & Kiểm tra trước khi nộp


In [ ]:
# [Z.1] Cay thu muc cuoi cung
import os

print(f"PROJECT: {CFG.PROJECT_DIR}")
print("="*60)
for goc, _, files in os.walk(CFG.PROJECT_DIR):
    level = goc.replace(CFG.PROJECT_DIR, "").count(os.sep)
    print(f"{'  '*level}{os.path.basename(goc) or '.'}/")
    for fn in sorted(files):
        sz = os.path.getsize(os.path.join(goc, fn))
        print(f"{'  '*(level+1)}{fn}  ({sz:,} B)")


In [ ]:
# [Z.2] Checklist truoc khi nop
checklist = [
    ("Y1", "spam_clean.csv tai pipeline_du_lieu",
        os.path.exists(f"{CFG.DIR_PIPELINE}/spam_clean.csv")),
    ("Y1", "vec.pkl + splits.pkl",
        os.path.exists(f"{CFG.DIR_PIPELINE}/vec.pkl") and
        os.path.exists(f"{CFG.DIR_PIPELINE}/splits.pkl")),
    ("Y2", "MLflow co 2 run (NB + XGB)",
        len(cli.search_runs([exp.experiment_id])) >= 2),
    ("Y3", "Registry co Production version",
        any("Production" in (getattr(v,"aliases",[]) or [])
            or getattr(v,"current_stage","")=="Production"
            for v in cli.search_model_versions(f"name='{CFG.MODEL_NAME}'"))),
    ("Y3", "registry_info.json",
        os.path.exists(f"{CFG.DIR_REGISTRY}/registry_info.json")),
    ("Y4", "API server file",      os.path.exists("/content/api_server.py")),
    ("Y4", "API duoc test",        os.path.exists(CFG.LOG_FILE)),
    ("Y5", "mlops.yml",
        os.path.exists(f"{CFG.DIR_OPS}/.github/workflows/mlops.yml")),
    ("Y5", "test_drift.py",
        os.path.exists(f"{CFG.DIR_OPS}/tests/test_drift.py")),
]

print("="*60)
print("  CHECKLIST NOP BAI")
print("="*60)
for tag, mota, ok in checklist:
    icon = "[v]" if ok else "[ ]"
    print(f"  {icon}  {tag}  {mota}")

so_xong = sum(1 for _, _, x in checklist if x)
print(f"\n  {so_xong}/{len(checklist)} muc da hoan thanh")
print("\n[NHO]: Chup man hinh MLflow UI + Swagger Docs (/docs) de nop kem")
